# 4주차 실습 1부 · 교재 PDF를 이미지로 바꾸기

교재 PDF에서 **교재 30~45면**만 골라 이미지 파일로 만듭니다.

- 교재 면 전체: 16장 (`교재32_전체.jpg`)
- 원본 한 면씩 자른 것: 30장 (`교재32_원본二二_오른쪽.jpg`, `교재32_원본二三_왼쪽.jpg`)

셀을 **위에서부터 차례로** 실행하세요. 셀 왼쪽의 ▶ 버튼을 누르면 됩니다.

## 셀 1 · 도구 설치
PDF를 다루는 도구(PyMuPDF)를 설치합니다. 몇 초 걸립니다.

In [ ]:
!pip -q install pymupdf

## 셀 2 · PDF 올리기
실행하면 **파일 선택** 버튼이 나옵니다. 필기가 없는 `2026년강의교재.pdf`를 고르세요.

(파일이 커서 올리는 데 1~2분 걸릴 수 있습니다.)

In [ ]:
from google.colab import files
uploaded = files.upload()
pdf_name = list(uploaded)[0]
print("올린 파일:", pdf_name)

## 셀 3 · 교재 30~45면을 이미지로 바꾸기

눈여겨볼 줄
- `교재면 + 4 - 1` — **PDF 쪽 = 교재 면 + 4**. 컴퓨터는 0부터 세므로 1을 뺍니다.
- `rotate(-90)` — 이 PDF는 장이 옆으로 누운 채 저장되어 있어서, 챗봇이 읽기 전에 똑바로 세웁니다.
- `원본면` — 교재 면과 원본 면의 대응표. 사람이 교재를 보고 확인해서 적은 데이터입니다.
- `겹침 = 20` — 가운데를 자를 때 첫 줄·마지막 줄이 잘리지 않도록 양쪽으로 조금 겹칩니다.

In [ ]:
import pymupdf, os
from PIL import Image

# 교재 면 : (오른쪽 원본 면, 왼쪽 원본 면)
원본면 = {
    30: ("서문", None),
    31: ("二〇", "二一"), 32: ("二二", "二三"), 33: ("二四", "二五"),
    34: ("二六", "二七"), 35: ("二八", "二九"), 36: ("三〇", "三一"),
    37: ("三二", "三三"), 38: ("六〇", "六一"), 39: ("六二", "六三"),
    40: ("六四", "六五"), 41: ("六六", "六七"),
    42: ("附錄一八", "附錄一九"), 43: ("附錄二〇", "附錄二一"),
    44: ("附錄二二", "附錄二三"), 45: ("附錄二四", "附錄二五"),
}

doc = pymupdf.open(pdf_name)
os.makedirs("교재이미지", exist_ok=True)
겹침 = 20   # 가운데를 자를 때 글자가 잘리지 않도록 양쪽으로 조금 겹친다

for 교재면, (오른쪽, 왼쪽) in 원본면.items():
    page = doc[교재면 + 4 - 1]            # PDF 쪽 = 교재 면 + 4 (0부터 세므로 1을 뺀다)
    pix = page.get_pixmap(dpi=200)
    img = Image.frombytes("RGB", (pix.width, pix.height), pix.samples)
    img = img.rotate(-90, expand=True)    # 옆으로 누운 장을 똑바로 세운다
    w, h = img.size
    img.save(f"교재이미지/교재{교재면}_전체.jpg", quality=90)
    if 왼쪽 is None:                      # 30면 서문은 자르지 않는다
        continue
    img.crop((w // 2 - 겹침, 0, w, h)).save(f"교재이미지/교재{교재면}_원본{오른쪽}_오른쪽.jpg", quality=90)
    img.crop((0, 0, w // 2 + 겹침, h)).save(f"교재이미지/교재{교재면}_원본{왼쪽}_왼쪽.jpg", quality=90)

print(len(os.listdir("교재이미지")), "장을 만들었습니다.")

## 셀 4 · 한 장 확인하기
글자가 똑바로 서 있는지, 첫 줄과 마지막 줄이 잘리지 않았는지 봅니다.

다른 면을 보려면 따옴표 안의 파일 이름만 바꾸세요. 예: `교재34_원본二六_오른쪽.jpg`

In [ ]:
from IPython.display import Image as 보기
보기("교재이미지/교재34_원본二六_오른쪽.jpg", width=400)

## 셀 5 · 한 번에 내려받기
46장을 zip 파일 하나로 묶어 내려받습니다.

⚠️ Colab 연결이 끊기면 만든 파일이 사라집니다. 꼭 내려받아 두세요.

In [ ]:
import shutil
shutil.make_archive("교재이미지", "zip", "교재이미지")
files.download("교재이미지.zip")